# Parts 2–3 · Dynamic CD8 and explicit anti-PD-1

Memory, Activated, Cytotoxic and Dysfunctional share a hazard kernel with the nonspatial solver. Every number in the default model is an assumption. Exposure is normalized, not a clinical dose.

In [ ]:
from pathlib import Path
from copy import deepcopy
import os, json, yaml
from vtsc.melanoma import DEFAULT_CONFIG
from vtsc.melanoma_results import run_melanoma, compare_melanoma
WORK = Path(os.environ.get("VTSC_NOTEBOOK_WORK", "outputs/notebook-v04"))
SEED = 21
HOURS = 12
config = deepcopy(DEFAULT_CONFIG)
config["model"]["schedule"] = [{"start_hour":2, "end_hour":96, "exposure":0.8}]
config_path = WORK / "assumptions.yaml"
if config_path.exists():
    raise FileExistsError("Use a new WORK directory; do not overwrite an accepted configuration")
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config["model"]["parameter_provenance"])


In [ ]:
summaries = {}
for arm in ["anti-pd1", "exposure-zero", "recognition-off", "transitions-off"]:
    summaries[arm] = run_melanoma(WORK / "input.zip", WORK / arm, config_path=config_path, arm=arm, hours=HOURS, seed=SEED)
print(json.dumps(summaries, indent=2))
comparison = compare_melanoma(WORK / "anti-pd1", WORK / "exposure-zero", WORK / "comparison")


In [ ]:
from vtsc.dynamics import solve, STATES
from vtsc.tissue_inputs import read_tissue_bundle
p = read_tissue_bundle(WORK / "input.zip")
initial = [sum(c["cd8_state"] == s for c in p["cells"]) for s in STATES]
context = {"contact":1.,"target":1.,"reactive":1.,"tumor_fraction":.5,"myeloid_fraction":.1}
expected = solve(initial, [0, 6, 12], config["model"], context)
print("Homogeneous hypothetical context, not a prediction for the spatial run:")
print(dict(zip(STATES, expected[-1])))


Compare multiple seeds with `vtsc ensemble-melanoma`. Recognition-off disables recognition, while exposure-zero preserves recognition and checkpoint inhibition. Checkpoint-off decouples the drug. No seed is selected by apparent benefit. Group-level variation is simulation uncertainty, not a patient confidence interval.